Dataset inventory

In [ ]:
from pathlib import Path
import os

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "Notebooks/01_dataset_inventory.ipynb").is_file())
os.chdir(root)

In [ ]:
import boto3
import pandas as pd
from pathlib import PurePosixPath
from collections import Counter
from IPython.display import display

BUCKET_NAME = "amal-stitchingnet-data-903473565738-ap-southeast-2-an"

REGION = "ap-southeast-2"

DATA_PREFIX = "data/"

s3 = boto3.client(
    "s3",
    region_name=REGION
)

print("S3 connection established")

response = s3.head_bucket(
    Bucket=BUCKET_NAME
)

print("Bucket accessible:", response is not None)

In [ ]:
paginator = s3.get_paginator("list_objects_v2")

all_objects = []

for page in paginator.paginate(
    Bucket=BUCKET_NAME,
    Prefix=DATA_PREFIX
):

    objects = page.get("Contents", [])

    all_objects.extend(objects)

print("Total files found:", len(all_objects))

In [ ]:
records = []

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".tif", ".tiff", ".webp"
}

for obj in all_objects:

    key = obj["Key"]

    if key.endswith("/"):
        continue

    path = PurePosixPath(key)

    relative_folders = path.parts[1:-1]

    records.append({

        "file_name": path.name,

        "s3_key": key,

        "extension": path.suffix.lower(),

        "file_size_kb": obj["Size"] / 1024,

        "last_modified": obj["LastModified"],

        "folder_1": (
            relative_folders[0]
            if len(relative_folders) > 0
            else None
        ),

        "folder_2": (
            relative_folders[1]
            if len(relative_folders) > 1
            else None
        ),

        "is_image": (
            path.suffix.lower() in IMAGE_EXTENSIONS
        )
    })

inventory_df = pd.DataFrame(records)

print("Inventory created")

display(inventory_df.head(10))

In [ ]:
print("Total objects:", len(inventory_df))

print(
    "Total images:",
    inventory_df["is_image"].sum()
)

print(
    "Non-image files:",
    (~inventory_df["is_image"]).sum()
)

print(
    "Total image storage (MB):",
    round(
        inventory_df.loc[
            inventory_df["is_image"],
            "file_size_kb"
        ].sum() / 1024,
        2
    )
)

In [ ]:
extension_counts = (
    inventory_df["extension"]
    .value_counts()
)

display(extension_counts)

In [ ]:
image_df = inventory_df[
    inventory_df["is_image"]
].copy()

fabric_distribution = (
    image_df.groupby("folder_1")
    .size()
    .sort_values(ascending=False)
)

display(
    fabric_distribution.to_frame(
        name="image_count"
    )
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(13, 6))

fabric_distribution.plot(
    kind="bar"
)

plt.title("Image Distribution Across Fabric Categories")

plt.xlabel("Fabric Category")

plt.ylabel("Number of Images")

plt.xticks(rotation=45, ha="right")

plt.tight_layout()

plt.show()

In [ ]:
folder_distribution = (
    image_df.groupby(
        ["folder_1", "folder_2"],
        dropna=False
    )
    .size()
    .reset_index(name="image_count")
    .sort_values("image_count", ascending=False)
)

display(folder_distribution.head(50))

In [ ]:
for key in image_df["s3_key"].head(40):

    print(key)

In [ ]:
from pathlib import Path

Path("reports/eda").mkdir(
    parents=True,
    exist_ok=True
)

local_report = "reports/eda/01_s3_inventory.csv"

inventory_df.to_csv(
    local_report,
    index=False
)

s3.upload_file(
    local_report,
    BUCKET_NAME,
    "eda/01_s3_inventory.csv"
)

print("Inventory report uploaded successfully")

In [ ]:
# Count images in each stitching class

class_distribution = (
    image_df.groupby("folder_2")
    .size()
    .reset_index(name="image_count")
)

# Extract numeric class identifier
class_distribution["class_id"] = (
    class_distribution["folder_2"]
    .str.extract(r"^(\d+)")
    .astype(int)
)

class_distribution = (
    class_distribution
    .sort_values("class_id")
    .reset_index(drop=True)
)

display(class_distribution)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.barh(
    class_distribution["folder_2"],
    class_distribution["image_count"]
)

plt.title("Stitching Class Distribution")

plt.xlabel("Number of Images")

plt.ylabel("Stitching Class")

plt.tight_layout()

plt.show()

In [ ]:
counts = class_distribution["image_count"]

print("Largest class:", counts.max())
print("Smallest class:", counts.min())

print(
    "Imbalance ratio:",
    round(counts.max() / counts.min(), 2)
)

print(
    "Total images:",
    counts.sum()
)

In [ ]:
fabric_class_table = pd.crosstab(
    image_df["folder_1"],
    image_df["folder_2"]
)

display(fabric_class_table)

In [ ]:
import re

name_parts = image_df["file_name"].str.extract(
    r"^([A-Z])_(\d{2})_(\d+)\.([A-Za-z0-9]+)$"
)

validation_df = pd.DataFrame({

    "filename_fabric": name_parts[0],

    "folder_fabric": image_df["folder_1"].str.extract(
        r"^([A-Z])\."
    )[0],

    "filename_class": pd.to_numeric(
        name_parts[1],
        errors="coerce"
    ),

    "folder_class": pd.to_numeric(
        image_df["folder_2"].str.extract(
            r"^(\d+)"
        )[0],
        errors="coerce"
    )
})

print(
    "Invalid filename patterns:",
    name_parts.isna().any(axis=1).sum()
)

print(
    "Fabric code mismatches:",
    (validation_df["filename_fabric"] !=
     validation_df["folder_fabric"]).sum()
)

print(
    "Class code mismatches:",
    (validation_df["filename_class"] !=
     validation_df["folder_class"]).sum()
)

In [ ]:
# Reorder columns by numerical class ID
class_order = (
    class_distribution
    .sort_values("class_id")["folder_2"]
    .tolist()
)

fabric_class_table = fabric_class_table.reindex(
    columns=class_order
)

# Convert counts into percentages within each fabric
fabric_class_percentages = (
    fabric_class_table
    .div(fabric_class_table.sum(axis=1), axis=0)
    * 100
)

display(fabric_class_percentages.round(1))

In [ ]:
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

# Select 2 random images from every stitching class
sample_df = (
    image_df
    .groupby("folder_2", group_keys=False)
    .sample(n=2, random_state=42)
)

sample_records = []

fig, axes = plt.subplots(
    nrows=11,
    ncols=2,
    figsize=(10, 32)
)

for row_idx, class_name in enumerate(class_order):

    class_samples = (
        sample_df[
            sample_df["folder_2"] == class_name
        ]
        .reset_index(drop=True)
    )

    for col_idx in range(2):

        row = class_samples.iloc[col_idx]

        # Download one image from S3
        response = s3.get_object(
            Bucket=BUCKET_NAME,
            Key=row["s3_key"]
        )

        image_bytes = response["Body"].read()

        # Open and decode image
        with Image.open(
            BytesIO(image_bytes)
        ) as original:

            original.load()

            width, height = original.size
            mode = original.mode
            image_format = original.format

            image = original.convert("RGB")

        # Display image
        ax = axes[row_idx, col_idx]

        ax.imshow(image)

        ax.set_title(
            f"{row['folder_1']}\n{row['file_name']}",
            fontsize=8
        )

        ax.axis("off")

        sample_records.append({
            "fabric": row["folder_1"],
            "class": class_name,
            "filename": row["file_name"],
            "width": width,
            "height": height,
            "mode": mode,
            "format": image_format
        })

plt.suptitle(
    "StitchingNet - Representative Images",
    fontsize=16
)

plt.tight_layout()

plt.show()

sample_metadata = pd.DataFrame(sample_records)

display(sample_metadata)